# Notebook 1 - Leitura e limpeza da base

Este notebook é responsável por toda a etapa de leitura, sendo o carregamento do arquivo bruto, reestruturação da matriz transposta em formato longo, classificação das colunas, tratamento de valores ausentes, detecção de outliers, cálculo da margem de segurança e estatística descritiva.

A saída é a base limpa df_limpo, gravada em disco e consumida pelos notebooks de normalização e gráficos.

## Configuração

A fonte de dados é o arquivo Data_Science_Datasets__Updated_07_08_.xlsx, planilha Option 3 - First cases HE. O carregamento aceita também a exportação em CSV utilizada nas versões anteriores dos notebooks, de modo que a mudança de fonte não quebre a execução em outro ambiente.

Cabe registrar que a leitura direta do .xlsx resolve a divergência de contagem observada anteriormente entre as seções da documentação: a exportação em CSV convertia oito células de percentual em texto, que era descartado como nulo, enquanto o .xlsx preserva os valores numéricos originais.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S1_S2"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os
import pandas as pd

pd.set_option("display.max_columns", None)

# Diretório onde as bases intermediárias são gravadas para os outros notebooks
DIR_SAIDA = "dados_intermediarios"
os.makedirs(DIR_SAIDA, exist_ok=True)

# Caminhos candidatos, em ordem de prioridade
CAMINHOS_CANDIDATOS = [
    "Data_Science_Datasets__Updated_07_08_.xlsx",
    "../../documents/outros/Data_Science_Datasets__Updated_07_08_.xlsx",
    "/mnt/project/Data_Science_Datasets__Updated_07_08_.xlsx",
    "../../documents/outros/dados_whirlpool.csv",
]

CAMINHO_ARQUIVO = next((c for c in CAMINHOS_CANDIDATOS if os.path.exists(c)), None)
if CAMINHO_ARQUIVO is None:
    raise FileNotFoundError(
        "Base não encontrada. Ajuste CAMINHOS_CANDIDATOS para o caminho local do arquivo."
    )

print("Fonte utilizada:", CAMINHO_ARQUIVO)

## Preparação da tabela

A planilha original é uma matriz transposta, em que cada coluna representa uma run de simulação e as primeiras linhas guardam os metadados de configuração daquela run, e não medições. As linhas seguintes correspondem aos termopares.

A reestruturação percorre três passos. 
Primeiro, os nomes das runs são tornados únicos, uma vez que alguns se repetem na planilha e a repetição quebraria o merge com os metadados, casando cada linha de sensor com todas as linhas de metadado de mesmo nome. Em seguida, o bloco de metadados é separado do bloco de sensores pela detecção automática da primeira linha preenchida na coluna de nomes. Por fim, a tabela é convertida para o formato longo, no qual cada linha corresponde a uma leitura de um termopar em uma run.

In [ ]:
%pip install openpyxl

In [ ]:
if CAMINHO_ARQUIVO.endswith(".csv"):
    df = pd.read_csv(CAMINHO_ARQUIVO, header=None)
else:
    df = pd.read_excel(CAMINHO_ARQUIVO, header=None)

# Nomes das runs
nome_run = df.iloc[0, 4:].tolist()


# Alguns nomes de run se repetem na planilha. Isso quebra o merge com os
# metadados mais abaixo (cada linha de sensor seria casada com TODAS as linhas
# de metadado com aquele nome, duplicando dados). Por isso, tornamos os nomes
# únicos antes de usá-los como chave.
def deduplicar(nomes):
    contagem = {}
    resultado = []
    for n in nomes:
        if n not in contagem:
            contagem[n] = 0
            resultado.append(n)
        else:
            contagem[n] += 1
            resultado.append(f"{n} ({contagem[n]})")
    return resultado


run_ids = deduplicar(nome_run)

# Detecta dinamicamente onde as linhas de metadado terminam e os sensores começam
mascara_preenchida = df.loc[1:, 1].notna()
primeira_linha_sensor = mascara_preenchida[mascara_preenchida].index[0]

print("Runs identificadas:", len(run_ids))
print("Primeira linha de sensor detectada:", primeira_linha_sensor)

In [ ]:
# Metadados de cada run
meta_labels = df.iloc[1:primeira_linha_sensor, 0].tolist()
valor_lista = df.iloc[1:primeira_linha_sensor, 4:].values
meta_df = pd.DataFrame(valor_lista.T, columns=meta_labels)

# A planilha intercala linhas de título de seção (ex.: "Mainback Components")
# entre os metadados. Elas não têm valor em nenhuma run e viram colunas vazias,
# que acabariam entrando no conjunto de features. São descartadas aqui.
colunas_vazias = [c for c in meta_df.columns if meta_df[c].isna().all()]
if colunas_vazias:
    meta_df = meta_df.drop(columns=colunas_vazias)
    print("Linhas de título descartadas dos metadados:", colunas_vazias)

meta_df.insert(0, "Run", run_ids)

# Linhas de sensores
body = df.iloc[primeira_linha_sensor:].copy()
sensor_rows = body[body[1].notna()].copy()
sensor_rows.columns = ["TC Number", "TC Name", "Key Area", "Active Limit"] + run_ids

# Preenche a área quando a célula está vazia (mesclada na planilha original)
sensor_rows["Key Area"] = sensor_rows["Key Area"].ffill()


def corrigir_percentual(valor):
    """Converte células gravadas com formato de percentual (ex.: '11377.00%')."""
    if isinstance(valor, str) and valor.strip().endswith("%"):
        return float(valor.strip().rstrip("%")) / 100
    return valor


sensor_rows[run_ids] = sensor_rows[run_ids].map(corrigir_percentual)

print("Termopares identificados:", len(sensor_rows))
print("Metadados por run:", meta_labels)

In [ ]:
# Melt: uma linha = um sensor em uma run
longo_df = sensor_rows.melt(
    id_vars=["TC Number", "TC Name", "Key Area", "Active Limit"],
    var_name="Run", value_name="Temperature"
)

longo_df["Temperature"] = pd.to_numeric(longo_df["Temperature"], errors="coerce")
longo_df["Active Limit"] = pd.to_numeric(longo_df["Active Limit"], errors="coerce")

longo_df = longo_df.merge(meta_df, on="Run", how="left")

print("Total de linhas antes da limpeza:", len(longo_df))
longo_df.head()

### Correções de erros encontrados na planilha original

Durante a investigação dos dados, foi encontrado um erro de formatação na planilha original que estava sendo tratado incorretamente como dado ausente. Oito células da coluna Run 5 - Broil Qualitative foram gravadas no Excel com formato de porcentagem, de modo que o valor 113,77 passou a ser exibido como 11377,00%. Sem correção, esse texto é convertido em NaN pela função de conversão numérica, e oito leituras reais de temperatura seriam descartadas. A correção foi feita pela divisão do valor por 100 e é preservada aqui para o caso de a base ser lida a partir da exportação em CSV.

Cabe registrar que a grafia Monitor (150 C), encontrada no Active Limit do sensor Turbo Duct / AC Blower Outlet, chegou a ser interpretada como um limite de segurança documentado, e o número foi extraído do texto. Conforme esclarecido pela parceira, esse valor constitui apenas uma referência de monitoramento, e não um limite. A extração foi, por isso, removida do carregamento, e toda grafia iniciada por Monitor passou a ser tratada como ausência de limite.

In [ ]:
print(longo_df.dtypes)  # Analisa os tipos dos dados. Auxilia a identificação de dados numéricos,
                        # já que eles são aqueles com os quais se pode realizar operações matemáticas.

### Reconstrução da coluna TC Number

In [ ]:
# Diagnóstico da coluna original, antes da reconstrução
tc_original = longo_df["TC Number"]
por_linha = longo_df.drop_duplicates(subset="TC Name")["TC Number"]

quebradas = por_linha.astype(str).str.contains("REF", na=False).sum()
vazias = por_linha.isna().sum()
validas = pd.to_numeric(por_linha, errors="coerce").notna().sum()

print("Composição da coluna TC Number original (por termopar):")
print(f"  identificadores válidos ...: {validas}")
print(f"  referências quebradas (#REF!): {quebradas}")
print(f"  em branco .................: {vazias}")
print(f"  total de termopares .......: {len(por_linha)}")

print("\nIdentificadores originais preservados:",
      sorted(pd.to_numeric(por_linha, errors="coerce").dropna().astype(int).tolist()))

# O TC Name é a chave que permanece íntegra e sustenta a reconstrução
assert longo_df.drop_duplicates(subset="TC Name")["TC Name"].notna().all()
print("TC Name sem repetições entre termopares:",
      longo_df["TC Name"].nunique() == len(por_linha))

In [ ]:
# Reconstrução com inteiro sequencial por termopar distinto, derivado do TC Name.
longo_df["TC Number"] = pd.factorize(longo_df["TC Name"])[0] + 1

print("TC Number reconstruído:", longo_df["TC Number"].dtype)
print("Faixa:", longo_df["TC Number"].min(), "a", longo_df["TC Number"].max())
print("Termopares distintos:", longo_df["TC Number"].nunique())

## Identificação das colunas numéricas e categóricas

Para a classificação de cada coluna, o tipo de dado retornado pelo dtypes foi consultado como ponto de partida, mas esse critério não é suficiente por si só: embora todos os objetos possam ser classificados como categóricos, nem todos os floats são numéricos, como ocorre no caso de um código identificador.

O critério final adotado na análise foi se a operação de média ou diferença entre os valores tem algum significado. Em caso afirmativo, a coluna foi classificada como numérica; caso contrário, como categórica.

Colunas numéricas:
- Temperature: temperatura medida pelo sensor em cada run. O cálculo de média e desvio padrão e a comparação entre valores possuem significado para o modelo preditivo a ser desenvolvido.

Colunas categóricas:
- Active Limit: limite de temperatura de referência do sensor. Ainda que seja expresso em graus Celsius, sua função é definir se a temperatura medida está acima ou abaixo do permitido, e não a de representar uma medição que varia a cada ensaio. Foi verificado que a coluna assume um número reduzido de valores distintos e que nenhum sensor possui mais de um valor, de modo que ela é inteiramente determinada pelo TC Name, constituindo um atributo fixo do componente definido por norma. Por essas razões, foi classificada como categórica.

- TC Name, Key Area: identificam o sensor e a área do fogão. São rótulos, e não quantidades.

- Run: identifica o teste realizado, sendo um rótulo, e não uma sequência numérica com significado matemático.

- Reason, Blower Direction, Housing, Board Position, Run Case, Losses, Deflector Hole, Material Coil Support: metadados de configuração de cada run, todos representando grupos ou categorias, e não grandezas.

- TC Number: apesar de ser um número, é um identificador do sensor, e não uma grandeza mensurável.

- Possui Active Limit: coluna adicionada posteriormente. Booleano que indica se existe um limite máximo permitido para a medição.

In [ ]:
colunas_numericas = ["Temperature"]
colunas_categoricas = [
    "TC Number", "TC Name", "Key Area", "Run", "Active Limit",
    "Reason", "Blower Direction", "Housing", "Board Position",
    "Run Case", "Losses", "Deflector Hole", "Material Coil Support"
]

print("Numéricas:", colunas_numericas)
print("Categóricas:", colunas_categoricas)

# Verificação: o Active Limit é constante dentro de cada termopar?
limites_por_sensor = longo_df.groupby("TC Name")["Active Limit"].nunique()
print("\nValores distintos de Active Limit na base:", longo_df["Active Limit"].nunique())
print("Sensores com mais de um Active Limit:", int((limites_por_sensor > 1).sum()))

## Preenchimento de valores ausentes nas colunas numéricas

### Temperature: razões para a remoção das linhas

A estratégia escolhida foi a eliminação das leituras ausentes (dropna), em vez do seu preenchimento. O padrão dos nulos foi investigado antes que a decisão fosse confirmada, e três achados sustentam a escolha.

O primeiro é que a maior parte dos nulos provém de sensores que nunca foram instrumentados. Os nulos não estão distribuídos aleatoriamente: cerca de dois terços concentram-se em apenas cinco sensores, que ficaram sem medição na maioria ou na totalidade das runs. Nesses casos não existe informação a ser recuperada, uma vez que o dado nunca chegou a ser medido. Qualquer imputação, seja por média, mediana ou interpolação, equivaleria a fabricar uma medição física que não ocorreu.

O segundo achado é que esses cinco sensores são justamente sensores de monitoramento, e não de segurança. Todos constam da lista sem_limite utilizada adiante, ou seja, correspondem aos sensores descritos pela parceira como não possuindo limite de segurança, monitorados porque seus valores influenciam algumas áreas do produto, sendo boa parte deles sensores de ar. Há coerência física em que sejam os menos instrumentados: trata-se de medições complementares de fluxo de ar, e não de verificações obrigatórias de segurança do componente.

O terceiro achado é que o descarte não desequilibra o conjunto de dados. A distribuição percentual das categorias foi comparada antes e depois do dropna, de modo a garantir que o conjunto de treino não fosse enviesado em favor de algum tipo de teste ou área do fogão.

In [ ]:
print(longo_df["Temperature"].isna().sum(), "de", len(longo_df))

# Concentração dos nulos por sensor
nulos_por_sensor = (
    longo_df.groupby("TC Name")["Temperature"]
    .apply(lambda s: s.isna().mean() * 100)
    .sort_values(ascending=False)
)
print("\nSensores com maior percentual de leituras ausentes:")
print(nulos_por_sensor.head(5).round(1).to_string())

In [ ]:
# Estratégia simples e lógica para a temperatura: retirar linhas com valores
# nulos, já que são inúteis para a análise e podem afetá-la indesejadamente.
df_limpo = longo_df.dropna(subset=["Temperature"]).copy()

print("Total de linhas em df_limpo:", len(df_limpo))
print("Sensores no conjunto:", df_limpo["TC Name"].nunique())

# Comparação da distribuição das categorias antes e depois do descarte,
# para confirmar que a base não foi desbalanceada.
for coluna in ["Run Case", "Key Area"]:
    antes = longo_df[coluna].value_counts(normalize=True) * 100
    depois = df_limpo[coluna].value_counts(normalize=True) * 100
    variacao = (depois - antes).abs().max()
    print(f"{coluna}: variação máxima de {variacao:.1f} ponto percentual")

### Sobre o sensor Oven RTD Connector Body

Todas as leituras do termopar Oven RTD Connector Body situam-se acima do seu Active Limit de 150 °C, sendo a menor delas 198,76 °C, o que representa um excedente de 48,76 °C sobre o limite. O comportamento é observado em todos os tipos de ensaio, o que afasta a hipótese de restrição a uma condição específica de teste e é compatível com falha de calibração ou com limite cadastrado incorretamente na planilha.

Por orientação da parceira, essas medições são desconsideradas. Como nenhuma das leituras do sensor é aproveitável, o descarte equivale à exclusão do próprio termopar do conjunto, que passa de 4065 para 4012 leituras e de 77 para 76 termopares.

Cabe registrar o impacto da decisão sobre a região Mainback, a única afetada, que passa de 610 leituras e 11 termopares, com média de 80,70 °C, para 557 leituras e 10 termopares, com média de 68,13 °C. As demais seis regiões permanecem inalteradas. Os valores dessa região apresentados no Anexo I precisam ser recalculados em decorrência da mudança.

In [ ]:
SENSOR_ANOMALO = "Oven RTD Connector Body"

anomalo = df_limpo[df_limpo["TC Name"] == SENSOR_ANOMALO]
if len(anomalo):
    limite = anomalo["Active Limit"].dropna().unique()
    acima = (anomalo["Temperature"] > anomalo["Active Limit"]).sum()
    print(f"{SENSOR_ANOMALO}: {len(anomalo)} leituras | Active Limit: {limite}")
    print(f"Leituras acima do limite: {acima} de {len(anomalo)}")
    print(f"Menor leitura registrada: {anomalo['Temperature'].min():.2f} °C")

# Desconsideramos o sensor que aparenta defeituoso
df_limpo = df_limpo[df_limpo["TC Name"] != SENSOR_ANOMALO].copy()

print("\nApós o descarte -> linhas:", len(df_limpo),
      "| termopares:", df_limpo["TC Name"].nunique())

## Preenchimento de valores ausentes nas colunas categóricas

Diferentemente da Temperature, os nulos nas colunas categóricas de metadado, como Losses e Reason, não decorrem de erro de leitura do sensor: correspondem a runs em que aquele dado simplesmente não foi preenchido na planilha original da Whirlpool. Por isso, em vez da remoção dessas linhas, o que descartaria leituras de temperatura válidas, esses nulos foram preenchidos com um rótulo explícito, de modo que os algoritmos de codificação categórica utilizados adiante no pré-processamento não sejam interrompidos ao encontrar NaN.

Também foram encontrados espaços em branco sobrando em alguns valores dessas colunas, como "Center " em vez de "Center", em Board Position. Trata-se de um erro grave para a codificação categórica: sem correção, "Center" e "Center " são tratados pelo Python como duas categorias diferentes, e o que deveria ser um único grupo acaba dividido em dois no One-Hot Encoding. A correção foi feita pela remoção dos espaços das pontas de todas as colunas categóricas antes da codificação.

### Active Limit: razões para o preenchimento

Para o Active Limit, foi comunicado pela parceira que os componentes sem Active Limit de fato não possuem limite, sendo apenas dados de monitoramento. Diferentemente da Temperature, aqui o nulo não significa medição perdida, e sim não aplicável. Por isso, em vez da remoção das linhas, uma coluna booleana auxiliar, Possui Active Limit, foi adicionada para informar se há um limite definido, e os nulos foram preenchidos com 0, evitando a perda de dados sem que fosse inventado um limite de segurança inexistente. Com o Active Limit tratado como variável categórica, o valor 0 passa a funcionar como a categoria pela qual são identificados os sensores sem limite aplicável.

In [ ]:
# Consolidação de categorias de Run Case escritas de formas diferentes
df_limpo["Run Case"] = df_limpo["Run Case"].replace("Broil Case 2", "Broil C2")
df_limpo["Run Case"] = df_limpo["Run Case"].replace("Broil Qual ", "Broil Qualitative")

# Componentes que não possuem limite de segurança, conforme a parceira
sem_limite = [
    "AC Blower Inlet Max", "AC Blower Inlet Avg", "Top Duct Inlet",
    "IPC Vertical Duct Inlet (Bottom)",
    "IPC Horizontal Duct Oulet (Inlet IPC Fan)", "IPC Housing Outlet",
    "IPC Blower (Air Outlet Temp)", "Heat Sink (Point/Probe)",
    "Dedicated IPC/CKTP Outlet", "Turbo Duct / AC Blower Outlet"
]

df_limpo["Possui Active Limit"] = ~df_limpo["TC Name"].isin(sem_limite)
df_limpo["Active Limit"] = df_limpo["Active Limit"].fillna(0)

print("Nulos de Active Limit antes do preenchimento:",
      longo_df["Active Limit"].isna().sum(), "de", len(longo_df))

# Verificação: a coluna deve ser exatamente equivalente a Active Limit != 0.
# Qualquer divergência indica contradição entre a lista e o valor recuperado.
divergencias = (df_limpo["Possui Active Limit"] != (df_limpo["Active Limit"] != 0)).sum()
print("Sensores sem limite aplicável:", (~df_limpo["Possui Active Limit"]).sum(), "linhas")
print("Divergências entre a lista e o Active Limit:", divergencias)

In [ ]:
colunas_categoricas_presentes = [c for c in colunas_categoricas if c in df_limpo.columns]
df_limpo[colunas_categoricas_presentes] = df_limpo[colunas_categoricas_presentes].fillna("Não Informado")

# Remove espaços em branco sobrando nas pontas (ex.: "Center " vira "Center"),
# que faziam o Python enxergar a mesma categoria como duas diferentes.
# TC Number e Active Limit são categóricos, mas guardados como número, por isso
# são excluídos do strip, que só se aplica a texto.
colunas_texto_presentes = [c for c in colunas_categoricas_presentes
                           if c not in ("TC Number", "Active Limit")]
df_limpo[colunas_texto_presentes] = df_limpo[colunas_texto_presentes].apply(lambda col: col.str.strip())

df_limpo["Blower Direction"] = df_limpo["Blower Direction"].str.strip().str.title()

print(df_limpo[colunas_categoricas_presentes].isna().sum())

## Tratamento dos outliers

Foi realizada a análise dos valores dos dados numéricos para a identificação dos outliers e, possivelmente, sua modificação, de modo que o modelo preditivo seja alimentado com dados relevantes e que não o enviesem. De início, já é possível determinar que apenas a temperatura deve ser tratada quanto a outliers, uma vez que o Active Limit é um limite fixo para cada peça, definido pela legislação.

Na primeira versão da detecção, os outliers eram agrupados apenas por TC Name, o que misturava leituras de Pyro, mais quente, e de Broil, mais variado, na mesma distribuição, fazendo com que leituras de Broil perfeitamente normais parecessem outliers apenas por comparação com Pyro. A correção foi feita pelo agrupamento por TC Name e Run Case em conjunto, de modo que cada leitura seja comparada apenas com leituras do mesmo sensor no mesmo tipo de teste.

A decisão foi a de manter todos os outliers detectados. Para os de Pyro, a justificativa está no próprio processo de autolimpeza pirolítica, o mais extremo de aquecimento do fogão. Para os de Broil, a coluna Reason desses casos foi verificada, e praticamente todos apresentam uma mudança de configuração documentada, como "Added a cover above Heatsink", "Part of study of position", "No cover on HS", "Double Flow on Blower" e "Baseline new Duct", ou seja, são variações intencionais do experimento, envolvendo cobertura, posição, vazão do soprador e duto, e não erro de medição.

Os boxplots que apoiam visualmente esta etapa encontram-se no notebook 3.

In [ ]:
def flag_outliers_iqr(serie):
    """Cálculo dos intervalos interquartis e marcação dos outliers."""
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    return (serie < limite_inferior) | (serie > limite_superior)


# Agrupamos por TC Name E Run Case (não só TC Name): Pyro e Broil aquecem o
# fogão de formas muito diferentes, então misturar os dois na mesma
# distribuição fazia leituras normais de Broil parecerem outliers só por
# comparação com Pyro, que é sistematicamente mais quente.
df_limpo["Temperature_outlier"] = (
    df_limpo.groupby(["TC Name", "Run Case"])["Temperature"].transform(flag_outliers_iqr)
)

print(df_limpo["Temperature_outlier"].sum(), "outliers de", len(df_limpo))
print("\nDistribuição por Run Case:")
print(df_limpo.loc[df_limpo["Temperature_outlier"], "Run Case"].value_counts().to_string())

# Todos os outliers são mantidos: a coluna auxiliar é apenas de diagnóstico.
df_limpo = df_limpo.drop(columns="Temperature_outlier")

## Margem de segurança (Active Limit - Temperature)

Ainda que o Active Limit seja tratado como variável categórica, seus valores permanecem registrados em graus Celsius, na mesma unidade da Temperature, o que permite uma comparação fisicamente direta entre as duas: o quão perto ou distante cada leitura se encontra do limite de segurança do próprio sensor.

Esta coluna constitui uma ferramenta de análise, e não uma feature do modelo. Como o alvo do modelo preditivo é a própria Temperature, a inclusão da margem em X configuraria vazamento de dados: sendo o Active Limit conhecido de antemão, a margem revela exatamente a temperatura que se pretende prever. O modelo acertaria quase tudo nos testes e falharia na prática, pois, para uma configuração nova, não existe margem calculável antes que a temperatura seja conhecida. Por essa razão, ela é calculada aqui, utilizada na análise descritiva e explicitamente removida de X no notebook 2.

O cálculo é feito nesta etapa, antes que a codificação seja aplicada, justamente porque a subtração exige que os valores do Active Limit ainda estejam em forma numérica. Após o One-Hot Encoding, a coluna é substituída por indicadores binários, e a operação deixa de ser possível. Pela mesma razão, o cálculo antecede a normalização da Temperature: com a padronização, a leitura deixaria de estar em graus Celsius e a diferença perderia o significado físico de graus até o limite.

Para os sensores que de fato não possuem limite de segurança, identificados por Possui Active Limit igual a False, a margem não existe fisicamente, pois não há limite do qual estar perto ou distante. Esses casos recebem NaN, e não 0, justamente para que não sejam confundidos com uma leitura real que tenha atingido exatamente o limite, situação em que a margem seria legitimamente zero.

In [ ]:
# Margem de Segurança: quantos graus faltam (ou passaram) até o limite do sensor.
# Calculada em °C, ANTES da normalização, para preservar o significado físico.
# ATENÇÃO: é coluna de análise, não entra em X (vazamento — ver markdown acima).
df_limpo["Margem de Seguranca"] = df_limpo["Active Limit"] - df_limpo["Temperature"]

# Sensores sem limite de segurança não têm margem fisicamente definida.
# Usamos NaN (e não 0) para não confundir "não aplicável" com uma leitura
# real que tenha atingido exatamente o limite.
df_limpo.loc[~df_limpo["Possui Active Limit"], "Margem de Seguranca"] = float("nan")

print(df_limpo["Margem de Seguranca"].describe())
print()
print("Leituras que ultrapassam o limite (margem negativa):",
      (df_limpo["Margem de Seguranca"] < 0).sum())
print("Sensores sem limite aplicável (margem NaN):",
      df_limpo["Margem de Seguranca"].isna().sum())

## Estatística descritiva das colunas

O objetivo desta etapa é compreender o comportamento da base de simulações térmicas da Whirlpool antes de alimentar o modelo preditivo.

A análise é dividida em duas partes. A primeira trata da variável numérica Temperature, cuja média, desvio padrão e dispersão têm significado direto para o problema, já que é a grandeza que o modelo será treinado para prever. A segunda trata das variáveis categóricas, que passam a incluir também o Active Limit: apesar de ser representado por um número, esse valor não é medido livremente, e sim tabelado, definido por norma técnica ou pela especificação de cada componente. Por isso, ele é analisado em termos de frequência e valores mais comuns, e não de média ou desvio padrão.

In [ ]:
print("ESTATÍSTICA DA VARIÁVEL NUMÉRICA:")
display(df_limpo[["Temperature"]].describe())

print("ESTATÍSTICAS DAS VARIÁVEIS CATEGÓRICAS:")
colunas_para_analise = ["TC Number", "TC Name", "Key Area", "Run",
                        "Reason", "Blower Direction", "Housing", "Board Position",
                        "Run Case", "Losses", "Deflector Hole", "Material Coil Support"]

df_categoricas_temp = df_limpo[colunas_para_analise].copy()
df_categoricas_temp["TC Number"] = df_categoricas_temp["TC Number"].astype(str)
df_categoricas_temp["Active Limit"] = df_limpo["Active Limit"].astype("category")

display(df_categoricas_temp.describe())

## Exportação da base limpa

A base limpa é gravada em disco para que os notebooks 2 e 3 partam exatamente do mesmo ponto, sem repetir a leitura e a limpeza. A coluna Possui Active Limit é gravada como booleano e restaurada como tal na leitura.

In [ ]:
CAMINHO_LIMPO = os.path.join(DIR_SAIDA, "df_limpo.csv")
df_limpo.to_csv(CAMINHO_LIMPO, index=False)

print("Base gravada em:", CAMINHO_LIMPO)
print("Dimensões:", df_limpo.shape)
print("Colunas:", list(df_limpo.columns))
df_limpo.head()